In [15]:
import pandas as pd
import os
import glob
from joblib import Parallel, delayed
from tqdm import tqdm

# ---------------------------#
#       CONFIGURATION        #
# ---------------------------#
os.chdir("/tmp")
input_folder = 'Individual_data_Uniti_CPU_parallel'
#input_folder = '/home/mdnl/myRDS/PabloRDS/Hamza/8gene/files'
output_folder = 'Individual_data_Uniti_generated_features_new'
#output_folder = '/home/mdnl/myRDS/PabloRDS/Hamza/Individual_data_Uniti_generated_features_cohort'

os.makedirs(output_folder, exist_ok=True)

# ---------------------------#
#   LOAD HELPER FILES        #
# ---------------------------#

gene_mapping = pd.read_csv('helper_files/gene_mapping.csv')
mapping_df = pd.read_csv('helper_files/unique_consequences_corresponding_with_imputed_CADDPHRED_Values_FIXED.csv')
gene_lengths = pd.read_csv('helper_files/gene_length_GRCh38_113.csv')

mapping_df.columns = mapping_df.columns.str.strip().str.replace(" ", "_")

# 🔬 CRITICAL FIX — Normalize Ensembl IDs in reference tables
gene_mapping['Ensembl_ID'] = gene_mapping['Ensembl_ID'].astype(str).str.split('.').str[0]
gene_lengths['gene_id'] = gene_lengths['gene_id'].astype(str).str.split('.').str[0]

# ---------------------------#
#     TARGET GENE FILTER     #
# ---------------------------#

TARGET_SYMBOLS = {
    "ELMOD1",
    "CIB2",
    "NPTN",
    "MGRN1",
    "COL9A2",
    "KLHDC7B",
    "GRXCR1",
    "MARVELD2",
}

TARGET_ENSEMBL_IDS = {
    "ENSG00000110675",  # ELMOD1
    "ENSG00000136425",  # CIB2
    "ENSG00000156642",  # NPTN
    "ENSG00000102858",  # MGRN1
    "ENSG00000049089",  # COL9A2
    "ENSG00000130487",  # KLHDC7B
    "ENSG00000215203",  # GRXCR1
    "ENSG00000152939",  # MARVELD2
}

# Keep Phenotype restricted to the same 8 genes
ensembl_id_set = set(TARGET_ENSEMBL_IDS)

# ---------------------------#
#     PROCESS FUNCTION       #
# ---------------------------#

def process_file(file_path):
    try:
        main_df = pd.read_csv(file_path, sep='\t', low_memory=False)

        # Drop rows without gene
        df_gene_cleaned = main_df.dropna(subset=['Gene']).copy()

        # 🔬 CRITICAL FIX — Normalize Ensembl IDs from VEP
        df_gene_cleaned['Gene'] = df_gene_cleaned['Gene'].astype(str).str.split('.').str[0]
        df_gene_cleaned['SYMBOL'] = df_gene_cleaned['SYMBOL'].astype(str).str.strip()

        # ---------------------------
        # Filter to target 8 genes
        # ---------------------------
        df_gene_cleaned = df_gene_cleaned[
            df_gene_cleaned['Gene'].isin(TARGET_ENSEMBL_IDS) |
            df_gene_cleaned['SYMBOL'].isin(TARGET_SYMBOLS)
        ].copy()

        if df_gene_cleaned.empty:
            return f"⚠️ Skipping {os.path.basename(file_path)}: No target genes found."

        # Ensure numeric
        df_gene_cleaned['CADD_PHRED'] = pd.to_numeric(df_gene_cleaned['CADD_PHRED'], errors='coerce')
        df_gene_cleaned['AF'] = pd.to_numeric(df_gene_cleaned['AF'], errors='coerce')

        # Define min/max CADD for this file
        min_cadd = df_gene_cleaned['CADD_PHRED'].min()
        max_cadd = df_gene_cleaned['CADD_PHRED'].max()

        # Build mapping dict (low/max replaced with file-specific min/max)
        mapping = {
            row['Unique_Null_Consequences']: (
                min_cadd if row['CADD_PHRED_imputed'] == 'low' else
                max_cadd if row['CADD_PHRED_imputed'] == 'max' else
                float(row['CADD_PHRED_imputed'])
            )
            for _, row in mapping_df.iterrows()
        }

        # Impute missing CADD_PHRED
        mask_missing = df_gene_cleaned['CADD_PHRED'].isna()
        df_gene_cleaned.loc[mask_missing, 'CADD_PHRED'] = (
            df_gene_cleaned.loc[mask_missing, 'Consequence'].map(mapping)
        )

        # ---------------------------
        # Merge with gene length
        # ---------------------------

        updated_dataset = df_gene_cleaned.merge(
            gene_lengths,
            how='left',
            left_on='Gene',
            right_on='gene_id'
        )

        # ----------------------------------
        # GENE SCORE CALCULATION
        # ----------------------------------

        frequency_disease = 0.001
        length_median = updated_dataset['gene_length'].median()

        gene_scores = (
            updated_dataset
            .groupby('gene_id', as_index=False)
            .agg(
                gene_score=('AF', lambda af: min((af.astype(float).prod() / frequency_disease), 1)),
                n_variants=('AF', 'count'),
                gene_length=('gene_length', 'first')
            )
        )

        gene_scores['gene_score'] = (
            gene_scores['n_variants']
            * (length_median / gene_scores['gene_length'])
            * gene_scores['gene_score']
        )

        gene_scores = gene_scores[['gene_id', 'gene_score']]

        updated_dataset = updated_dataset.merge(gene_scores, on='gene_id', how='left')

        # ---------------------------
        # Phenotype features
        # ---------------------------

        updated_dataset['Phenotype'] = updated_dataset['Gene'].isin(ensembl_id_set).astype(int)

        ph_counts = (
            updated_dataset[updated_dataset['Phenotype'] == 1]
            .groupby('SYMBOL')
            .size()
        )

        updated_dataset['PH'] = updated_dataset['SYMBOL'].map(ph_counts).fillna(0).astype(int)

        # ---------------------------
        # Save file
        # ---------------------------

        output_file = os.path.join(output_folder, os.path.basename(file_path))
        updated_dataset.to_csv(output_file, sep='\t', index=False)

        return f"✅ Processed {os.path.basename(file_path)}"

    except Exception as e:
        return f"❌ Failed {os.path.basename(file_path)}: {e}"


# ---------------------------#
#   PARALLEL FILE PROCESS    #
# ---------------------------#

tsv_files = glob.glob(os.path.join(input_folder, "Uniti_case_*.tsv"))
import os
os.chdir("/tmp")
results = Parallel(n_jobs=-1)(
    delayed(process_file)(f) for f in tqdm(tsv_files, desc="Processing files", unit="file")
)

print("\n".join(results))
print(f"\n🎉 All files processed and saved in '{output_folder}'")

Processing files: 100%|██████████████████| 166/166 [00:00<00:00, 15227.00file/s]


✅ Processed Uniti_case_ZU69.tsv
✅ Processed Uniti_case_GR24.tsv
✅ Processed Uniti_case_ZU86.tsv
✅ Processed Uniti_case_PT115.tsv
✅ Processed Uniti_case_PT207.tsv
✅ Processed Uniti_case_PT51.tsv
✅ Processed Uniti_case_GR115.tsv
✅ Processed Uniti_case_ZU09.tsv
✅ Processed Uniti_case_ZU36.tsv
✅ Processed Uniti_case_GR150.tsv
✅ Processed Uniti_case_GR66.tsv
✅ Processed Uniti_case_PT105.tsv
✅ Processed Uniti_case_GR06.tsv
✅ Processed Uniti_case_PT121.tsv
✅ Processed Uniti_case_ZU54.tsv
✅ Processed Uniti_case_GR75.tsv
✅ Processed Uniti_case_GR33.tsv
✅ Processed Uniti_case_GR2.tsv
✅ Processed Uniti_case_GR49.tsv
✅ Processed Uniti_case_PT25.tsv
✅ Processed Uniti_case_ZU051.tsv
✅ Processed Uniti_case_GR23.tsv
✅ Processed Uniti_case_PT101.tsv
✅ Processed Uniti_case_ZU22.tsv
✅ Processed Uniti_case_GR34.tsv
✅ Processed Uniti_case_ZU29.tsv
✅ Processed Uniti_case_ZU42.tsv
✅ Processed Uniti_case_GR81.tsv
✅ Processed Uniti_case_PT156.tsv
✅ Processed Uniti_case_ZU21.tsv
✅ Processed Uniti_case_ZU44.tsv


In [16]:
# ---------------------------#
#   VERIFY ONE OUTPUT FILE   #
# ---------------------------#

import random

output_files = glob.glob(os.path.join(output_folder, "Uniti_case_*.tsv"))

if len(output_files) == 0:
    print("\n⚠️ No output files found to inspect.")
else:
    sample_file = random.choice(output_files)  # pick any patient randomly
    print(f"\n📂 Inspecting sample file: {os.path.basename(sample_file)}")

    try:
        df_sample = pd.read_csv(sample_file, sep="\t", low_memory=False)

        print("\nColumns created:")
        print(df_sample.columns.tolist())

        print("\nFirst 5 rows:")
        print(df_sample.head())

        print("\nFeature summary:")
        important_cols = ['CADD_PHRED','gene_length','gene_score','Phenotype','PH']
        existing_cols = [c for c in important_cols if c in df_sample.columns]
        print(df_sample[existing_cols].describe(include='all'))

        # Check merge success
        print("\nMerge success check:")
        print("gene_length non-null %:",
              round(df_sample['gene_length'].notna().mean()*100,2), "%")
        print("gene_score non-null %:",
              round(df_sample['gene_score'].notna().mean()*100,2), "%")

    except Exception as e:
        print(f"❌ Failed to read sample file: {e}")



📂 Inspecting sample file: Uniti_case_GR21.tsv

Columns created:
['CHROM', 'POS', 'REF', 'ALT', 'Gene', 'SYMBOL', 'Consequence', 'CADD_PHRED', 'AF', 'GT', 'gene_id', 'start', 'end', 'gene_length', 'gene_score', 'Phenotype', 'PH']

First 5 rows:
   CHROM       POS REF ALT             Gene    SYMBOL            Consequence  \
0   chr1  40307451   G   C  ENSG00000049089    COL9A2       missense_variant   
1   chr4  42893406   C   T  ENSG00000215203    GRXCR1       missense_variant   
2   chr4  43030452   G   A  ENSG00000215203    GRXCR1       missense_variant   
3   chr5  69419483   C   T  ENSG00000152939  MARVELD2       missense_variant   
4  chr16   4614411   T  TG  ENSG00000102858     MGRN1  upstream_gene_variant   

   CADD_PHRED        AF   GT          gene_id     start       end  \
0      17.460  0.001166  0/1  ENSG00000049089  40300489  40317813   
1      22.500  0.001166  0/1  ENSG00000215203  42892713  43030658   
2      32.000  0.001166  0/1  ENSG00000215203  42892713  43030658  